## 1. Setup & Imports

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_colwidth', 200)
plt.style.use('seaborn-v0_8-whitegrid')

# Add src to path
import sys
sys.path.append('../src')

print("✅ Setup complete!")

## 2. Data Loading & Exploration

In [ ]:
# Load the dataset
df = pd.read_csv('../data/jobs.csv')

print(f"📊 Dataset Shape: {df.shape}")
print(f"\n📋 Columns: {df.columns.tolist()}")
print(f"\n📈 Data Types:")
print(df.dtypes)

In [ ]:
# Display first few rows
print("🔍 First 5 rows of the dataset:")
df.head()

In [ ]:
# Basic statistics
print("📊 Dataset Statistics:\n")
print(f"Total job postings: {len(df):,}")
print(f"Unique job titles: {df['Job Title'].nunique():,}")
print(f"Missing values:")
print(df.isnull().sum())

In [ ]:
# Job description length analysis
df['desc_word_count'] = df['Job Description'].apply(lambda x: len(str(x).split()))
df['desc_char_count'] = df['Job Description'].apply(lambda x: len(str(x)))

print("📝 Job Description Length Statistics:")
print(f"Average word count: {df['desc_word_count'].mean():.0f}")
print(f"Median word count: {df['desc_word_count'].median():.0f}")
print(f"Min word count: {df['desc_word_count'].min()}")
print(f"Max word count: {df['desc_word_count'].max()}")

In [ ]:
# Top job titles
print("🏆 Top 15 Job Titles:")
df['Job Title'].value_counts().head(15)

## 3. Text Preprocessing

In [ ]:
# Import preprocessing module
from preprocess import TextPreprocessor, preprocess_dataframe

# Initialize preprocessor
preprocessor = TextPreprocessor(
    lowercase=True,
    remove_punctuation=True,
    remove_numbers=True,
    remove_stopwords=True,
    lemmatize=True
)

print("✅ Preprocessor initialized!")

In [ ]:
# Example preprocessing
sample_text = df['Job Description'].iloc[0]
print("📄 Original Text:")
print(sample_text[:500])
print("\n" + "="*50 + "\n")
print("🧹 Preprocessed Text:")
print(preprocessor.preprocess(sample_text)[:500])

In [ ]:
# Preprocess entire dataset (using sample for speed)
sample_df = df.head(1000).copy()
sample_df = preprocess_dataframe(sample_df, preprocessor=preprocessor)

print("✅ Preprocessing complete!")
sample_df[['Job Title', 'Job Description', 'cleaned_text']].head(3)

## 4. Skill Extraction

In [ ]:
# Import skill extraction module
from skill_extractor import (
    RuleBasedExtractor, 
    MLBasedExtractor, 
    HybridSkillExtractor,
    extract_skills_from_dataframe,
    get_skill_statistics,
    PREDEFINED_SKILLS
)

print(f"📚 Predefined skills count: {len(PREDEFINED_SKILLS)}")
print(f"\n📝 Sample skills: {list(PREDEFINED_SKILLS)[:20]}")

In [ ]:
# Rule-based extraction example
rule_extractor = RuleBasedExtractor()
sample_desc = df['Job Description'].iloc[0]

print("🔍 Rule-Based Extraction Example:")
print(f"\nJob Title: {df['Job Title'].iloc[0]}")
print(f"\nExtracted Skills: {rule_extractor.extract(sample_desc)}")

In [ ]:
# ML-based extraction example
ml_extractor = MLBasedExtractor()

print("🤖 ML-Based Extraction Example:")
print(f"\nExtracted Skills: {ml_extractor.extract(sample_desc)}")

In [ ]:
# Hybrid extraction on sample
print("🔄 Extracting skills from job descriptions...")
skills_df = extract_skills_from_dataframe(sample_df, method='hybrid')

print("\n✅ Skill extraction complete!")
skills_df[['Job Title', 'Extracted Skills', 'Skills Count']].head(10)

In [ ]:
# Skill statistics
stats = get_skill_statistics(skills_df)

print("📊 Skill Extraction Statistics:")
print(f"\nTotal unique skills found: {stats['total_unique_skills']}")
print(f"Total skill mentions: {stats['total_skill_mentions']}")
print(f"\n🏆 Top 10 Skills:")
for skill, count in stats['top_10_skills']:
    print(f"  {skill}: {count}")

## 5. Text Summarization

In [ ]:
# Import summarization module
from summarizer import TextRankSummarizer, HybridSummarizer, summarize_dataframe

# Initialize TextRank summarizer
textrank = TextRankSummarizer()

print("✅ Summarizer initialized!")

In [ ]:
# TextRank summarization example
sample_desc = df['Job Description'].iloc[0]

print("📄 Original Description:")
print(sample_desc[:800])
print("\n" + "="*50)
print("\n📝 TextRank Summary:")
print(textrank.summarize(sample_desc, num_sentences=3))

In [ ]:
# Summarize sample dataset
print("🔄 Generating summaries for sample dataset...")
summary_df = summarize_dataframe(sample_df.head(100), method='textrank', num_sentences=2)

print("\n✅ Summarization complete!")
summary_df[['Job Title', 'Summary']].head(5)

## 6. Evaluation

In [ ]:
# Import evaluation module
from evaluate import (
    evaluate_extraction,
    evaluate_extractor_quality,
    compare_extraction_methods,
    print_comparison_summary,
    generate_evaluation_report
)

print("✅ Evaluation module loaded!")

In [ ]:
# Merge skills with original data for evaluation
eval_df = skills_df.copy()

# Evaluate extraction quality
print("📊 Evaluating skill extraction quality...")
metrics = evaluate_extractor_quality(eval_df)

print("\n" + "="*50)
print("EVALUATION RESULTS")
print("="*50)
print(f"\nMacro Precision: {metrics['macro_precision']:.4f}")
print(f"Macro Recall: {metrics['macro_recall']:.4f}")
print(f"Macro F1-Score: {metrics['macro_f1']:.4f}")
print(f"\nMicro Precision: {metrics['micro_precision']:.4f}")
print(f"Micro Recall: {metrics['micro_recall']:.4f}")
print(f"Micro F1-Score: {metrics['micro_f1']:.4f}")

In [ ]:
# Compare extraction methods
print("🔄 Comparing extraction methods...")
comparison_df = compare_extraction_methods(sample_df.head(50))
print_comparison_summary(comparison_df)

## 7. Visualizations

In [ ]:
# Import visualization module
from viz import (
    plot_job_title_distribution,
    create_wordcloud,
    plot_skill_frequency,
    plot_skills_per_job,
    plot_description_length_distribution
)

print("✅ Visualization module loaded!")

In [ ]:
# Job title distribution
print("📊 Job Title Distribution:")
fig = plot_job_title_distribution(df, top_n=15)
plt.show()

In [ ]:
# Word cloud
print("☁️ Word Cloud of Job Descriptions:")
fig = create_wordcloud(df['Job Description'].tolist()[:5000])
plt.show()

In [ ]:
# Skill frequency
print("📈 Most Frequent Skills:")
fig = plot_skill_frequency(skills_df, top_n=20)
plt.show()

In [ ]:
# Skills per job distribution
print("📊 Skills per Job Distribution:")
fig = plot_skills_per_job(skills_df)
plt.show()

In [ ]:
# Description length distribution
print("📏 Job Description Length Distribution:")
fig = plot_description_length_distribution(df)
plt.show()

## 8. Save Results

In [ ]:
import os

# Create output directory
os.makedirs('../output', exist_ok=True)

# Save extracted skills (full dataset)
print("💾 Processing full dataset for final output...")

# Extract skills from full dataset
full_skills_df = extract_skills_from_dataframe(df, method='hybrid')

# Save extracted skills
skills_output = full_skills_df[['id', 'Job Title', 'Extracted Skills']]
skills_output.to_csv('../output/extracted_skills.csv', index=False)
print("✅ Saved: ../output/extracted_skills.csv")

In [ ]:
# Summarize and save (sample for speed)
print("📝 Generating summaries...")
summary_output_df = summarize_dataframe(df.head(1000), method='textrank')
summary_output = summary_output_df[['id', 'Job Title', 'Summary']]
summary_output.to_csv('../output/job_summary.csv', index=False)
print("✅ Saved: ../output/job_summary.csv")

In [ ]:
# Final summary
print("\n" + "="*50)
print("🎉 ANALYSIS COMPLETE!")
print("="*50)
print(f"\n📊 Total jobs analyzed: {len(df):,}")
print(f"🔍 Skills extracted for: {len(full_skills_df):,} jobs")
print(f"📝 Summaries generated for: {len(summary_output_df):,} jobs")
print(f"\n📁 Output files:")
print("  - ../output/extracted_skills.csv")
print("  - ../output/job_summary.csv")